# Prepare Data

## Pinyin Mapping

In [1]:
INITIALS = [
    "", "b","p","m","f","d","t","n","l","g","k","h",
    "j","q","x","zh","ch","sh","r","z","c","s"
]

FINALS = [
    "a","ai","an","ang","ao","e","ei","en","eng","er",
    "i","ia","ian","iang","iao","ie","in","ing","iong","iu",
    "o","ong","ou",
    "u","ua","uai","uan","uang","ui","un","uo",
    "v","ve","van","vn"
]

TONES = ["1","2","3","4","5"]

ZERO_INITIAL_MAP = {
    # y-series
    "yi":"i",
    "ya":"ia",
    "yao":"iao",
    "ye":"ie",
    "you":"iu",
    "yan":"ian",
    "yin":"in",
    "yang":"iang",
    "ying":"ing",
    "yong":"iong",

    "yu":"v",
    "yue":"ve",
    "yuan":"van",
    "yun":"vn",

    # w-series
    "wu":"u",
    "wa":"ua",
    "wo":"uo",
    "wai":"uai",
    "wei":"ui",
    "wan":"uan",
    "wen":"un",
    "wang":"uang",
    "weng":"ong",
}


def normalize_zero_initial(base):
    return ZERO_INITIAL_MAP.get(base, base)


def normalize_jqx(final):
    if final.startswith("u"):
        mapping = {
            "u": "v",
            "ue": "ve",
            "uan": "van",
            "un": "vn",
        }
        return mapping.get(final, final)
    return final


init2idx = {v:i for i,v in enumerate(INITIALS)}
final2idx = {v:i for i,v in enumerate(FINALS)}
tone2idx = {v:i for i,v in enumerate(TONES)}

idx2init = {i:v for v,i in init2idx.items()}
idx2final = {i:v for v,i in final2idx.items()}
idx2tone = {i:v for v,i in tone2idx.items()}


def decompose_pinyin(token: str):
    tone = token[-1]

    if not tone.isdigit():
        tone = "5" # neutral tone, usually has no digit
        base = token
    else:
        base = token[:-1]

    base = normalize_zero_initial(base)

    # longest-match for initials

    try:
        for ini in sorted(INITIALS, key=len, reverse=True):
            if base.startswith(ini):
                final = base[len(ini):]

                if ini in {"j","q","x"}:
                    final = normalize_jqx(final)
                
                return [init2idx[ini], final2idx[final], tone2idx[tone]]
    except Exception:
        raise ValueError("Invalid pinyin token")

## Char Mapping

In [2]:
import pandas as pd

dataset = pd.read_csv("")


def align(token):
    if token[-1].isalpha():  # e.g. de -> de5
        return token + '5'
    return token


import pickle as pkl

with open("", "rb") as f:
    embedding_map = pkl.load(f)


# Actual filtration
dataset = dataset[
    dataset['pinyin'].str.split().transform(lambda x: all(align(y) in embedding_map.keys() for y in x))
]

In [3]:
import numpy as np
import torch

all_pinyin_tokens = list(embedding_map.keys())

pinyin2idx = {tok: i for i, tok in enumerate(all_pinyin_tokens)}
embedding_matrix = torch.tensor(np.stack([embedding_map[tok] for tok in all_pinyin_tokens]), dtype=torch.float32)

In [4]:
from collections import Counter

def build_char_vocab(df, min_freq=1, add_pad=True, add_unk=True):
    """
    Build character vocabulary from the 'text' column.
    Returns char2idx and idx2char.
    """
    counter = Counter()
    for text in df['text']:
        counter.update(text)
    # Filter by min_freq if needed
    chars = [ch for ch, cnt in counter.items() if cnt >= min_freq]
    chars = sorted(chars)
    vocab = []
    if add_pad:
        vocab.append('<PAD>')
    if add_unk:
        vocab.append('<UNK>')
    vocab.extend(chars)
    char2idx = {ch: i for i, ch in enumerate(vocab)}
    idx2char = vocab
    return char2idx, idx2char

char2idx, idx2char = build_char_vocab(dataset)
print(f"Vocabulary size: {len(char2idx)}")

Vocabulary size: 5464


# Prepare Model

## Load Model

In [5]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn

# Device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load BERT tokenizer and model
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
bert_model = BertModel.from_pretrained("bert-base-chinese")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import torch.nn as nn

class ClassifierHead(nn.Module):
    """MLP with one hidden layer for classification."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [7]:
class BertPinyinEncoder(nn.Module):
    def __init__(self, bert_model, num_initials, num_finals, num_tones, mlp_hidden_dim=256, dropout=0.3):
        super().__init__()
        self.bert = bert_model
        self.initial_head  = ClassifierHead(768, mlp_hidden_dim, num_initials, dropout)
        self.final_head    = ClassifierHead(768, mlp_hidden_dim, num_finals, dropout)
        self.tone_head     = ClassifierHead(768, mlp_hidden_dim, num_tones, dropout)
        self.phonetic_head = ClassifierHead(768, mlp_hidden_dim, 1024, dropout)

    def forward(self, input_ids, attention_mask, return_hidden=False):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state   # (batch, seq_len, 768)
        logits_i = self.initial_head(hidden)
        logits_f = self.final_head(hidden)
        logits_t = self.tone_head(hidden)

        if return_hidden:
            phonetic = self.phonetic_head(hidden)
            return logits_i, logits_f, logits_t, phonetic
        return logits_i, logits_f, logits_t

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Decoder(nn.Module):
    """
    Decodes character‑level hidden representations (from an encoder) into character indices.
    Optionally applies a BiLSTM on top of the input hidden states.
    """
    def __init__(self, input_dim, hidden_dim, num_layers, vocab_size_char, mlp_hidden_dim=128, dropout=0.3):
        """
        Args:
            input_dim: dimension of input hidden states (e.g., encoder hidden_dim * 2)
            hidden_dim: LSTM hidden size
            num_layers: number of LSTM layers
            vocab_size_char: size of character vocabulary
            mlp_hidden_dim: hidden dimensionality of MLP classfication head
            dropout: dropout probability (applied after LSTM)
        """
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            bidirectional=True, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.head = ClassifierHead(1024, mlp_hidden_dim, vocab_size_char, dropout)

    def forward(self, hidden_states, char_lengths):
        """
        Args:
            hidden_states: (batch, max_chars, input_dim) - padded character‑level representations
            char_lengths: (batch,) - actual lengths of each sequence (for packing)
        Returns:
            logits: (batch, max_chars, vocab_size_char) - unnormalized scores for each character
        """
        # Pack for variable‑length sequences
        packed = pack_padded_sequence(
            hidden_states, char_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        lstm_out = self.dropout(lstm_out)
        logits = self.head(lstm_out)
        return logits

In [9]:
class BertEncoderDecoderPipeline(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=False):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask, char_lengths, return_encoder_output=False):
        logits_i, logits_f, logits_t, hidden = self.encoder(input_ids, attention_mask, return_hidden=True)
        char_logits = self.decoder(hidden, char_lengths)
        if return_encoder_output:
            return char_logits, (logits_i, logits_f, logits_t, hidden)
        return char_logits

    def get_hidden(self, input_ids, attention_mask, char_lengths):
        *_, hidden = self.encoder(input_ids, attention_mask, return_hidden=True)
        return hidden

In [10]:
# Instantiate model
num_initials = len(init2idx)
num_finals   = len(final2idx)
num_tones    = len(tone2idx)

encoder = BertPinyinEncoder(bert_model, num_initials, num_finals, num_tones).to(device)

In [11]:
input_dim = 1024
hidden_dim = 512
mlp_hidden_dim = 256
num_layers = 4
vocab_size_char = len(idx2char)

decoder = Decoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    vocab_size_char=vocab_size_char,
    mlp_hidden_dim=mlp_hidden_dim
).to(device)

In [12]:
# Load the saved state dict
encoder.load_state_dict(torch.load('', map_location=device))
encoder.to(device)

print("Loaded encoder successfully!")

Loaded encoder successfully!


In [13]:
# Load the saved state dict
decoder.load_state_dict(torch.load('', map_location=device))
decoder.to(device)

print("Loaded encoder successfully!")

Loaded encoder successfully!


In [14]:
# Create pipeline with frozen encoder
pipeline = BertEncoderDecoderPipeline(encoder, decoder, freeze_encoder=True)
pipeline.to(device)

print("Created pipeline successfully")

Created pipeline successfully


## Inference

In [15]:
def predict_characters(sentence, pipeline, tokenizer, idx2char, device):
    """
    Predict the character sequence for a given Chinese sentence using the trained pipeline.

    Args:
        sentence: string of Chinese characters (e.g., "你好世界")
        pipeline: trained EncoderDecoderPipeline (in eval mode)
        tokenizer: BERT tokenizer (e.g., BertTokenizer)
        idx2char: list mapping index -> character (including <UNK> etc.)
        device: torch device

    Returns:
        predicted_chars: string of predicted characters
    """
    # Tokenize without adding special tokens ([CLS], [SEP])
    encoding = tokenizer(sentence, add_special_tokens=False, return_tensors='pt')
    input_ids = encoding['input_ids']           # (1, seq_len)
    attention_mask = encoding['attention_mask'] # (1, seq_len)
    char_lengths = torch.tensor([input_ids.size(1)], dtype=torch.long)

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    char_lengths = char_lengths.to(device)

    with torch.no_grad():
        logits = pipeline(input_ids, attention_mask, char_lengths)  # (1, seq_len, vocab_char)

    pred_indices = logits.argmax(dim=-1).squeeze(0).cpu().numpy()   # (seq_len,)
    predicted_chars = ''.join([idx2char[idx] for idx in pred_indices])
    return predicted_chars

In [16]:
def predict_characters_topk(
    sentence,
    pipeline,
    tokenizer,
    idx2char,
    device,
    k=5
):
    """
    Returns:
        predicted_chars : str
        topk_indices    : np.ndarray (seq_len, k)
    """

    encoding = tokenizer(
        sentence,
        add_special_tokens=False,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']

    char_lengths = torch.tensor(
        [input_ids.size(1)],
        dtype=torch.long
    )

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    char_lengths = char_lengths.to(device)

    with torch.no_grad():

        logits = pipeline(
            input_ids,
            attention_mask,
            char_lengths
        )

    pred_indices = (
        logits.argmax(dim=-1)
        .squeeze(0)
        .cpu()
        .numpy()
    )

    predicted_chars = ''.join(
        idx2char[idx]
        for idx in pred_indices
    )

    topk_indices = (
        torch.topk(
            logits,
            k=k,
            dim=-1
        )
        .indices
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return predicted_chars, topk_indices

In [17]:
predict_characters("纽约时报现年八十二岁的阿尔及利亚強人总统阿卜杜勒阿齐兹布特弗利卡月底将下台", pipeline, bert_tokenizer, idx2char, device)

'纽约时报现年八十二岁的阿尔吉利亚删人总统阿维杜勒阿奇兹布特福利卡月底将下台'

# Benchmarking

## Load Dataset

In [18]:
perturbation_df = pd.read_csv("")

In [19]:
perturbation_df.head()

,position,original_char,replacement_char,original_word,perturbed_word,word_freq,perturbed_freq
0,0,中,忠,中国,忠国,20800,0
1,1,个,各,一个,一各,14404,0
2,0,美,每,美国,每国,11073,0
3,0,自,字,自己,字己,8939,0
4,0,可,渴,可能,渴能,8367,0


## Evaluation

In [20]:
total = 0

# =====================================================
# Phrase-level decomposition
# =====================================================

phrase_full_recovery = 0
phrase_full_copy = 0
phrase_partial_recovery = 0
phrase_target_copy_other_errors = 0
phrase_novel_target = 0

# =====================================================
# Target decomposition
# =====================================================

target_recovery = 0
target_copy = 0
target_novel = 0

# =====================================================
# Non-target decomposition
# =====================================================

nontarget_preserve = 0
nontarget_change = 0

# =====================================================
# Top-k
# =====================================================

top3_correct = 0
top5_correct = 0

# =====================================================
# Examples
# =====================================================

N_EXAMPLES = 20

correct_examples = []
incorrect_examples = []

In [21]:
from tqdm import tqdm

for _, row in tqdm(
    perturbation_df.iterrows(),
    total=len(perturbation_df)
):

    original_word = row["original_word"]
    perturbed_word = row["perturbed_word"]

    prediction, topk = predict_characters_topk(
        perturbed_word,
        pipeline,
        bert_tokenizer,
        idx2char,
        device,
        k=5
    )

    total += 1

    # =====================================================
    # Target positions
    # =====================================================

    target_positions = [
        pos
        for pos in range(len(original_word))
        if original_word[pos] != perturbed_word[pos]
    ]

    if len(target_positions) == 0:
        continue

    target_states = []

    # =====================================================
    # Target decomposition
    # =====================================================

    for pos in target_positions:

        pred_char = (
            prediction[pos]
            if pos < len(prediction)
            else None
        )

        if pred_char == original_word[pos]:

            target_recovery += 1
            target_states.append("recovery")

        elif pred_char == perturbed_word[pos]:

            target_copy += 1
            target_states.append("copy")

        else:

            target_novel += 1
            target_states.append("novel")

    # =====================================================
    # Non-target decomposition
    # =====================================================

    nontarget_changed = False

    for pos in range(len(original_word)):

        if pos in target_positions:
            continue

        pred_char = (
            prediction[pos]
            if pos < len(prediction)
            else None
        )

        if pred_char == original_word[pos]:

            nontarget_preserve += 1

        else:

            nontarget_change += 1
            nontarget_changed = True

    # =====================================================
    # Phrase decomposition
    # =====================================================

    all_recovered = all(
        state == "recovery"
        for state in target_states
    )

    all_copied = all(
        state == "copy"
        for state in target_states
    )

    any_novel = any(
        state == "novel"
        for state in target_states
    )

    if prediction == original_word:

        phrase_full_recovery += 1

        if len(correct_examples) < N_EXAMPLES:
            correct_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })

    elif prediction == perturbed_word:

        phrase_full_copy += 1

        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })

    elif all_recovered and nontarget_changed:

        phrase_partial_recovery += 1

        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })

    elif all_copied and nontarget_changed:

        phrase_target_copy_other_errors += 1

        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })

    elif any_novel:

        phrase_novel_target += 1

        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })

    # =====================================================
    # Top-k
    # =====================================================

    recovered_top3 = True
    recovered_top5 = True

    for pos, true_char in enumerate(original_word):

        top3_chars = [
            idx2char[idx]
            for idx in topk[pos][:3]
        ]

        top5_chars = [
            idx2char[idx]
            for idx in topk[pos][:5]
        ]

        if true_char not in top3_chars:
            recovered_top3 = False

        if true_char not in top5_chars:
            recovered_top5 = False

    if recovered_top3:
        top3_correct += 1

    if recovered_top5:
        top5_correct += 1

100%|██████████| 2598/2598 [02:44<00:00, 15.83it/s]


In [22]:
import pandas as pd

phrase_table = pd.DataFrame({
    "Metric": [
        "Full Recovery",
        "Full Copy",
        "Partial Recovery",
        "Target Copy + Other Errors",
        "Novel Target"
    ],
    "Fraction": [
        phrase_full_recovery / total,
        phrase_full_copy / total,
        phrase_partial_recovery / total,
        phrase_target_copy_other_errors / total,
        phrase_novel_target / total
    ]
})

target_total = (
    target_recovery +
    target_copy +
    target_novel
)

target_table = pd.DataFrame({
    "Metric": [
        "Recovery",
        "Copy",
        "Novel"
    ],
    "Fraction": [
        target_recovery / target_total,
        target_copy / target_total,
        target_novel / target_total
    ]
})

nontarget_total = (
    nontarget_preserve +
    nontarget_change
)

nontarget_table = pd.DataFrame({
    "Metric": [
        "Preserve",
        "Change"
    ],
    "Fraction": [
        nontarget_preserve / nontarget_total,
        nontarget_change / nontarget_total
    ]
})

topk_table = pd.DataFrame({
    "Metric": [
        "Top-3 Recovery",
        "Top-5 Recovery"
    ],
    "Fraction": [
        top3_correct / total,
        top5_correct / total
    ]
})

print(f"Samples: {total:,}")

print("\n=== Phrase-Level Decomposition ===")
display(phrase_table)

print("\n=== Target Position Decomposition ===")
display(target_table)

print("\n=== Non-Target Position Decomposition ===")
display(nontarget_table)

print("\n=== Top-k Recovery ===")
display(topk_table)

Samples: 2,598

=== Phrase-Level Decomposition ===


,Metric,Fraction
0,Full Recovery,0.337567
1,Full Copy,0.023095
2,Partial Recovery,0.096228
3,Target Copy + Other Errors,0.035412
4,Novel Target,0.507698



=== Target Position Decomposition ===


,Metric,Fraction
0,Recovery,0.433795
1,Copy,0.058507
2,Novel,0.507698



=== Non-Target Position Decomposition ===


,Metric,Fraction
0,Preserve,0.493841
1,Change,0.506159



=== Top-k Recovery ===


,Metric,Fraction
0,Top-3 Recovery,0.441493
1,Top-5 Recovery,0.479215


In [23]:
from IPython.display import HTML, display

def color_prediction(original, prediction):

    output = []

    max_len = max(
        len(original),
        len(prediction)
    )

    for i in range(max_len):

        true_char = (
            original[i]
            if i < len(original)
            else ""
        )

        pred_char = (
            prediction[i]
            if i < len(prediction)
            else ""
        )

        if true_char == pred_char:
            color = "green"
        else:
            color = "red"

        output.append(
            f'<span style="color:{color};font-weight:bold">'
            f'{pred_char}'
            '</span>'
        )

    return ''.join(output)

In [24]:
html = "<h2>Correct Recoveries</h2>"

for ex in correct_examples:

    html += f"""
    <div style="margin-bottom:12px">
      Original:&nbsp;&nbsp;{ex['original']}<br>
      Perturbed:&nbsp;{ex['perturbed']}<br>
      Prediction:&nbsp;{color_prediction(ex['original'], ex['prediction'])}
    </div>
    """

display(HTML(html))

In [25]:
html = "<h2>Incorrect Recoveries</h2>"

for ex in incorrect_examples:

    html += f"""
    <div style="margin-bottom:12px">
      Original:&nbsp;&nbsp;{ex['original']}<br>
      Perturbed:&nbsp;{ex['perturbed']}<br>
      Prediction:&nbsp;{color_prediction(ex['original'], ex['prediction'])}
    </div>
    """

display(HTML(html))